In [1]:
import pandas as pd
import os
import ast 
import numpy as np
from dotenv import load_dotenv
from elasticsearch import Elasticsearch

load_dotenv()  # Load API key from .env file

INDEX_NAME = "vectorhood"
OS_HOST = os.getenv("OS_HOST")
OS_API_KEY = os.getenv("OS_API_KEY")

client = Elasticsearch(
    hosts=[OS_HOST],
    api_key=OS_API_KEY
)


#### Get the range of values in the embedded vectors

In [2]:

# Convert the 'Embedding' column to a 2D NumPy array
df = pd.read_csv("out/clothing_items.csv")
df["Embedding"] = df["Embedding"].apply(ast.literal_eval)
    
embedding_matrix = np.array(df['Embedding'].tolist())

overall_min = embedding_matrix.min()
overall_max = embedding_matrix.max()

print("🔽 Overall lowest value:", overall_min)
print("🔼 Overall highest value:", overall_max)

norms = np.linalg.norm(embedding_matrix, axis=1)

print("Average vector norm:", norms.mean())
print("Min norm:", norms.min())
print("Max norm:", norms.max())



🔽 Overall lowest value: -0.21069336
🔼 Overall highest value: 0.21044922
Average vector norm: 0.9999940313129656
Min norm: 0.9996328519347859
Max norm: 1.0003698451726455


### The embedding vectors are normalized and lay on the n dimentional unit sphere!

Now I want to understand how they actually capture the meaning of the text.
In order to do that, I want to query the embedding vectors and see if I can make sense of the results.

## Lets check if there is a dimention that correspons with some reasonable feature like "Animlness" or "Fruitness"

If it was so, we would exepect animals or fruit to give a high similarity score (1) to the corresponding dimension and a low score (0) to the other dimensions.

In [3]:
DIM = 384
max_similarity_dim_index = 0
max_similarity_dim_val = 0

vector_sim_along_dim = [0.0] * (DIM)
for i in range(DIM):
    print(f"Processing dimension {i+1}/{DIM}...")
    vector_pos_x = [0.0] * (DIM)
    vector_neg_x = [0.0] * (DIM)
    
    vector_pos_x[i] = 1.0
    vector_neg_x[i] = -1.0

    def knn_search(query_vector, top_k=1):
        body = {
            "knn": {
                "field": "embedding",
                "query_vector": query_vector,
                "k": top_k,
                "num_candidates": 100
            }
        }

        response = client.search(index=INDEX_NAME, body=body)
        return response['hits']['hits']

    # Search for each vector
    for doc in knn_search(vector_pos_x):
        vector_sim_along_dim[i] = doc['_score']

    for doc in knn_search(vector_neg_x):
        if doc['_score'] > vector_sim_along_dim[i]:
            vector_sim_along_dim[i] = doc['_score']
            
# Calculate the min, max and mean similarity
min_similarity_dim_val = min(vector_sim_along_dim)
max_similarity_dim_val = max(vector_sim_along_dim)
mean_similarity_dim_val = sum(vector_sim_along_dim) / len(vector_sim_along_dim)

print(f"Dim {i} - Min similarity: {min_similarity_dim_val}")
print(f"Dim {i} - Max similarity: {max_similarity_dim_val}")
print(f"Dim {i} - Mean similarity: {mean_similarity_dim_val}")

Processing dimension 1/384...
Processing dimension 2/384...
Processing dimension 3/384...
Processing dimension 4/384...
Processing dimension 5/384...
Processing dimension 6/384...
Processing dimension 7/384...
Processing dimension 8/384...
Processing dimension 9/384...
Processing dimension 10/384...
Processing dimension 11/384...
Processing dimension 12/384...
Processing dimension 13/384...
Processing dimension 14/384...
Processing dimension 15/384...
Processing dimension 16/384...
Processing dimension 17/384...
Processing dimension 18/384...
Processing dimension 19/384...


KeyboardInterrupt: 

We can see moderate similariry on all dimensions.
This hints at the fact that modern embeddings don't capture meaning in a single dimension.
There's no dimension that corresponds to a single feature like "animalness" or "fruitness". 
But rather, the embeddings are a combination of many features that are not easily interpretable in a single dimension.

Then maybe there are multi dimnstional features that are captured by the embeddings?